# 4. AMBIT + TCSPC: phi2 from intensity and from lifetime

This notebook drives the AMBIT and the Felix TDC (the TCSPC card) together, to measure phi2 both from the AMBIT fluorescence intensity and from the fluorescence lifetime.

The TCSPC excitation laser is the 488 nm module. Set it to 50 % power and an 80 MHz repetition rate. It cannot be controlled from software (its trigger could be gated through the EXT SYNC IN mode, but that is out of scope for now).

The SPAD detector saturates above about 3 Mcps (million counts per second) and recovers from saturation within microseconds.

The Felix records TCSPC histograms for the whole run, which follows this schedule:

| Step | Duration | AMBIT measuring light (100 Hz) | AMBIT actinic light |
|---|---|---|---|
| 1. Laser only | 1 s | off | off |
| 2. Dark reference | 1 s | on | off |
| 3. Saturating light (Fm) | about 0.83 s | on | 255 (maximum), with 4 dark pauses |
| 4. Dark | 1 s | on | off |
| 5. Laser only | 1 s | off | off |

During the actinic light (the Fm measurement) the SPAD sees about 20 Mcps, so it saturates. To measure the lifetime, the actinic light is interrupted 4 times, for about 11 ms every 200 ms. In each pause the SPAD is no longer saturated, so the Felix can record an Fm decay histogram.

The histograms are fitted with an IRF reconvolution model, and the lifetimes (tau) are reported for each of the 4 pauses.


In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))      # run the notebook from scripts/ambit-gate
import importlib, ambit_gate as ag, ambit_chop as ac, blank_dose as bd, reconv as rc, ambit_pause as ap
ag, ac, bd, rc, ap = (importlib.reload(m) for m in (ag, ac, bd, rc, ap))   # edits to the modules apply on re-run

RUNS_DONE = globals().get("RUNS_DONE", [])
RUNS = Path("runs")
IRF = bd.load_irf()    # first CSV in irf-library/ (None if missing)


In [ ]:
PORT = None            # e.g. "COM23" to skip the port scan
LASER_HZ = 80e6        # laser mode selector
THRESHOLD_V = 1.2      # SYNC and SPAD are 2.5 V LVTTL

for _name in ("ambit", "felix"):
    try:
        globals()[_name].close()     # re-running must not leave a port locked
    except (KeyError, NameError):
        pass
ambit = ag.find_ambit(PORT)
felix = ag.FelixLink().open()
felix.set_thresholds(THRESHOLD_V)
print("Felix input rates:", {k: f"{v:,.0f}/s" for k, v in felix.count_rates(0.5).items()})

## 1. The run

**Before you start**
- The leaf is in place, the laser is on and unblocked. Write the laser power and the ND in `NOTE`.
- No dark adaptation is needed. The actinic-0 block comes first and is the reference: the leaf
  under the laser alone, with the AMBIT measuring pulses at 50 Hz, like in the pauses.


In [ ]:
LEVEL = 255                    # actinic byte of the lit blocks
DARK_POINTS = 100              # x: AMBIT points with the LED off before the first block and after the last (1 s at 100 Hz)
BLOCK_S = 0.2                  # light before each pause
PAUSE_AMBIT_PULSES = PAUSE_SAMPLES = (1, 1, 1, 1)   # one entry per pause: AMBIT points in it (1 point ~ 9.6 ms dark at 100 Hz)
FREQ = 100                     # AMBIT measuring rate, every line
WINDOW_S = 0.005               # Felix histogram window
SESSION = RUNS / (time.strftime("%Y%m%d_%H%M%S") + "_pauses")
NOTE = ""              # what is set up, e.g. "laser 20 %, ND on the SPAD, leaf piece 5 mm"

# dark (DARK_POINTS) | lit BLOCK_S, pause | ... x len(PAUSE_AMBIT_PULSES) | dark (DARK_POINTS)
PROTOCOL = ap.train_protocol(LEVEL, DARK_POINTS, BLOCK_S, PAUSE_AMBIT_PULSES, FREQ)
print(ap.describe(PROTOCOL))

LABEL = "home-leaf"
RUNS_DONE = globals().get("RUNS_DONE", [])
d = ap.run_pauses(ambit, felix, SESSION / f"pauses_{LABEL}_{time.strftime('%H%M%S')}", PROTOCOL,
                  window_s=WINDOW_S, note=NOTE, threshold_v=THRESHOLD_V, laser_hz=LASER_HZ)
RUNS_DONE.append(d)
res = ap.analyse_pauses(d, irf=IRF)
print(ap.report(res))
ap.plot_pauses(res)

### Lifetime fits of this run

**Main fit: IRF reconvolution** (`reconv.py`). Every decay is fitted as a whole, rise included,
with two exponentials convolved with the measured IRF (`IRF`, the first file in `irf-library/`):

m(t) = floor + IRF(t - shift) (*) [a1 exp(-t/tau1) + a2 exp(-t/tau2)], periodic over the 12.5 ns
laser period.

- **Window:** from 1.5 ns before the laser peak to 7 ns after it (1.0 to 9.5 ns on the axis), so
  the echo ~8 ns after the peak is left out.
- **Free parameters:** both lifetimes and amplitudes, the floor, and the IRF shift. The shift is
  needed because the IRF was measured on scattered laser light, and the laser timing drifts by
  tens of ps during a run.
- **Method:** Poisson maximum likelihood; the errors come from the fit covariance.

**What is reported for each component.** Its lifetime; its **height at t0**, the counts per 47 ps
bin of the component (convolved with the IRF) at the laser peak t0, as drawn in the figure (the two
heights plus the floor add up to the fit at t0); and its **share of the photons** (a x tau, over
the whole decay). The
amplitude before the IRF (a, counts per bin at time zero, not normalised) is given only for a
**resolved** component, one with tau >= half the IRF FWHM (0.15 ns here). For a shorter component
the IRF smears the decay so much that a cannot be measured: the same photons fit equally well
with a twice as large and tau twice as short, so a comes out far above the data and means nothing.
Only its photon count is determined.

**Lifetime reported.** `t_int` = photon-weighted mean lifetime of all components,
sum(a tau^2) / sum(a tau): the model's counterpart of the model-free `t_mean` (t_mean also contains
the IRF delay after its peak, so it is a little longer). The amplitude-weighted average `t_avg` is
no longer shown: the fast component (0.05-0.25 ns depending on the run; scatter or PSI, the IRF
cannot tell) got up to ~40 % of its weight while holding ~10 % of the photons, and t_avg jumped to
tau2 whenever that component fell below the resolution limit. `tau2` is the lifetime of the slow,
resolved component and is what `phi2_lifetime` uses.

There is one figure per group: laser only (reference, before and after the run), the pauses after
actinic 100, the pauses after actinic 250. Each panel shows the data, the fit (solid in the
window, thin outside), and each component convolved with the IRF, drawn from zero under the data
(blue: component 1, purple: component 2), with the floor dashed. The dotted line is t0, the laser
peak: there the dots of the two components plus the floor add up to the fit (black dot), and the
text box writes the sum out. The slow component has its own maximum later than t0, so read the
heights on the dotted line. The axis is linear so that heights add up by eye; each panel has its
own range. `ap.plot_fits(res, yscale="log", ylim=(1, 1e5))` shows the tails on one log axis
instead (there heights do not add by eye). The residuals in sigma are below (grey outside the
window). The figures are saved as `pauses_fits_*.png`.

The third figure of `ap.plot_pauses(res)` puts every pause ("Fm n", n = pause number) against the decay before the run, laser
only ("Fo"): the pause's raw data as points with its fit, and Fo as light grey points with its fit
behind it, no components. Both are in counts per second per 47 ps bin, raw counts divided by their
acquisition time, because a pause lasts 25-90 ms and Fo about 1 s. The legend gives tau2 of both and
phi2_lifetime = (tau2,Fm - tau2,Fo) / tau2,Fm, with its error from the two tau2 errors.
Saved as `pauses_fits_vs_before.png`; `ap.plot_pauses(res, ref="reference")` compares with the
actinic-0 line instead. `ap.plot_tail_fits(res, IRF)` draws the old tail fits.

## 3. Re-analyse a saved run

In [ ]:
# the last run of this session, else the newest saved run that has data (aborted runs leave empty folders)
RUN = RUNS_DONE[-1] if RUNS_DONE else sorted(p for p in RUNS.glob("*_pauses/pauses_*") if (p / "felix.npz").exists())[-1]
res = ap.analyse_pauses(RUN, irf=IRF)
print(ap.report(res))
ap.plot_pauses(res)
print(ap.fit_report(res))
ap.plot_fits(res, IRF);

## 4. Light series: pre-illumination, then the measurement

Per level X of `PRE_LEVELS`, in order:

1. A one-sample arrun at X with the persist flag switches the AMBIT LED on (one measuring flash) and leaves it on while the AMBIT is idle for `T_WAIT_S`: no more measuring flashes. The laser stays on the spot, as always. The SPAD rate is printed at the start and the end of the wait, so a LED that did not stay on shows up.
2. The LED is switched off (a one-sample dark arrun) and the measurement of section 1 runs without its 1 s laser-only pre- and post-record (0.2 s Felix margins instead): 1 s dark line with the measuring light at 100 Hz, the 255 train with its 4 pauses, 1 s dark line. So the leaf is in the dark for about 1.5 s before the first 255 block and 1 s after the last one.
3. Straight on to the next level, no rest.

Interrupting the kernel stops the series and switches the LED off; the runs done so far stay analysable (`series.json` is updated after every run).

**Per level:** AMBIT Fo = mean s/r of the last `FO_POINTS` dark points before the 255 train (0.5 s at 100 Hz), Fm = mean of the last `FM_POINTS` lit points of the train (20 per 0.2 s block, 80 in all), phi2 = (Fm - Fo) / Fm, with the error from the scatter of the points. tau2 and t_int of the reference line (Fo, the 1 s dark line) and of every pause (Fm). phi2_lifetime = (tau2,Fm - tau2,Fo) / tau2,Fm per pause, 4 values, and their mean and SD.

**Figures** (`series_phi2.png` in the session folder, with `series.csv` and `series_results.json`): phi2 against the pre-illumination (AMBIT, and the mean of the 4 lifetime values); tau2 of Fo (left axis) and of the 4 Fm pauses (right axis, the mean joined) against the pre-illumination; and phi2_lifetime against the AMBIT phi2, the mean of the 4 pauses as the marker (bar: SD) with the 4 values as faint dots and the 1:1 line.

In [ ]:
PRE_LEVELS = (0, 10, 20, 40, 80, 100, 160)   # pre-illumination actinic bytes, in run order (0: laser only)
T_WAIT_S = 60                              # LED on, AMBIT idle (no measuring flashes), before each measurement
FO_POINTS = 50                              # AMBIT Fo: mean of the last 50 dark points (0.5 s) before the 255 train
FM_POINTS = 50                              # AMBIT Fm: mean of the last 50 lit points of the train (20 per block, 80 in all)
SERIES = RUNS / (time.strftime("%Y%m%d_%H%M%S") + "_light")
NOTE = ""              # what is set up, e.g. "laser 50 %, no ND, leaf piece 5 mm"

PROTOCOL = ap.train_protocol(LEVEL, DARK_POINTS, BLOCK_S, PAUSE_AMBIT_PULSES, FREQ)   # the measurement of section 1
print(f"{len(PRE_LEVELS)} levels x ({T_WAIT_S} s wait + ~4 s measurement): about {len(PRE_LEVELS) * (T_WAIT_S + 4) / 60:.0f} min")
dirs = ap.run_light_series(ambit, felix, SERIES, LABEL, PRE_LEVELS, T_WAIT_S, PROTOCOL,
                           window_s=WINDOW_S, note=NOTE, threshold_v=THRESHOLD_V, laser_hz=LASER_HZ)
ser = ap.analyse_light_series(SERIES, irf=IRF, fo_points=FO_POINTS, fm_points=FM_POINTS)
print(ap.series_report(ser))
ap.plot_light_series(ser)
# a saved series (also an interrupted one): ser = ap.analyse_light_series(RUNS / "<date>_light", irf=IRF); ap.plot_light_series(ser)
# every run of it in full: for res in ser["results"]: print(ap.report(res)); ap.plot_pauses(res, ref="reference")

In [ ]:
# for res in ser["results"]: print(ap.report(res)); ap.plot_pauses(res, ref="reference")

In [ ]:
# Re-analyse the newest light series on disk (needs only the import cell, not the AMBIT or the Felix).
# An interrupted series works too: series.json lists the runs that finished.
SERIES = sorted(p for p in RUNS.glob("*_light") if (p / "series.json").exists())[-1]
print(SERIES)
ser = ap.analyse_light_series(SERIES, irf=IRF, fo_points=globals().get("FO_POINTS", ap.FO_POINTS),
                              fm_points=globals().get("FM_POINTS", ap.FM_POINTS))
print(ap.series_report(ser))
ap.plot_light_series(ser)

In [ ]:
ambit.close()
felix.close()